# Die Quellen von `gnd_context`: was drinsteht, was ankommt, was fehlt

Dieses Notebook zeigt, **was** in jeder Quelle steht, was [`build.py`](src/ie_agent/gnd/build.py) daraus übernimmt und **wie viel** davon beim Agenten in [`gnd_context`](src/ie_agent/gnd/context.py#L73-L133) ankommt. Gebaut wird alles einmal mit `ie-agent build-gnd` ([build_gnd, build.py:228-263](src/ie_agent/gnd/build.py#L228-L263)). Zur Laufzeit lädt [Gnd.load, graph.py:63-75](src/ie_agent/gnd/graph.py#L63-L75) die fertigen Pickles.

```
Quelle 1  vocab/GND-Subjects-plus.ttl            ─┐
Quelle 2  gnd_graph/authorities-gnd-*_lds.ttl.gz  ├─ build.py ─► graph.pkl + details.pkl ─┐
Quelle 3  gnd_graph/gnd-sc.ttl                   ─┘                                        ├─► gnd_context
          corpora/ground-truth.arrow (nur Training) ─── graph.py: Häufigkeit ──────────────┤
          state des Agenten (gelesen, abgebildet) ── zur Laufzeit ─────────────────────────┘
```

| | Datei | Herkunft | liefert für `gnd_context` | gelesen in |
|---|---|---|---|---|
| **1** | `vocab/GND-Subjects-plus.ttl` | KI-FS Prompt (DVC) | ID, Name, Ober-/Unter-/verwandte Begriffe | [_parse_graph, build.py:48-95](src/ie_agent/gnd/build.py#L48-L95) |
| **2** | `authorities-gnd-sachbegriff_lds.ttl.gz`, `authorities-gnd-geografikum_lds.ttl.gz` | DNB Open Data (CC0) | Definition, Benutzungshinweis, Beschreibung, Sachgruppen-Code, Art des Oberbegriffs | [_read_dumps, build.py:115-153](src/ie_agent/gnd/build.py#L115-L153), [_parse_details, build.py:156-188](src/ie_agent/gnd/build.py#L156-L188) |
| **3** | `gnd-sc.ttl` | DNB (GND-Sachgruppen) | Name zu jedem Sachgruppen-Code | [_category_names, build.py:102-112](src/ie_agent/gnd/build.py#L102-L112) |
| (+) | `corpora/ground-truth.arrow` | eigenes Gold, nur Trainingssplit | wie oft ein Begriff im Training vergeben wurde | [training_frequency, graph.py:30-34](src/ie_agent/gnd/graph.py#L30-L34) |

Wo die Dateien liegen, steht in `config.yaml` (`paths.gnd_ttl`, `gnd.dump_urls`, `gnd.classification_url`). Die Pfade baut [gnd_files, build.py:195-205](src/ie_agent/gnd/build.py#L195-L205).

**Aufbau:**
- Teil 1–3: jede Quelle einzeln, roh und in Zahlen
- Teil 4: das Trainings-Gold
- Teil 5: die fertige Antwort für E-Learning, jede Zeile mit ihrer Quelle
- Teil 6: Statistik, was jede Quelle beiträgt, was vollständig ist und was fehlt

Die Zellen mit den DNB-Dumps lesen jeweils beide Dateien komplett, das dauert jeweils etwa 10–30 Sekunden.

In [1]:
import gzip, json, re, sys
from collections import Counter
from pathlib import Path

import pandas as pd
import pyarrow.ipc

ROOT = Path("/home/harbc/ki-fsprompt")
sys.path.insert(0, str(ROOT / "ie_agnet/src"))
from ie_agent.config import read_config
from ie_agent.gnd import Gnd, gnd_context
from ie_agent.gnd.build import _category_names, nfc

VOCAB = ROOT / "vocab/GND-Subjects-plus.ttl"
GNDDIR = ROOT / "ie_agnet/gnd_graph"
DUMPS = {"Sachbegriff": GNDDIR / "authorities-gnd-sachbegriff_lds.ttl.gz",
         "Geografikum": GNDDIR / "authorities-gnd-geografikum_lds.ttl.gz"}
SC = GNDDIR / "gnd-sc.ttl"

cfg = read_config()
gnd = Gnd.load(cfg)                    # graph.pkl + details.pkl + Häufigkeit im Training
g, details = gnd.g, gnd.details
name_zu_id = {n: i for i, n in g["pref"].items()}
ALLE = set(g["pref"])
N = len(ALLE)

def anteil(n, von=N):
    return f"{n:,} ({n / von:.1%})".replace(",", ".")

BEISPIEL = name_zu_id["E-Learning"]
print(f"{N:,} Begriffe geladen · Beispiel: E-Learning = {BEISPIEL}".replace(",", "."))

309.417 Begriffe geladen · Beispiel: E-Learning = 967279453


## Teil 1 · Quelle 1: das Vokabular `vocab/GND-Subjects-plus.ttl`

Die Datei stammt aus KI-FS Prompt und ist dort die Grundlage des Mapping-Schritts (Weaviate). Sie ist im **SKOS**-Format: ein Block pro Begriff. Gelesen wird sie von [_parse_graph, build.py:48-95](src/ie_agent/gnd/build.py#L48-L95): Datei in Blöcke teilen (Leerzeile), Block in Eigenschaften teilen (`;` am Zeilenende), dann pro Eigenschaft ein Regex.

### 1.1 Roh: der Block von E-Learning

In [2]:
text_vokabular = VOCAB.read_text(encoding="utf-8")
start = text_vokabular.index(f"gnd:{BEISPIEL} a skos:Concept")
print(text_vokabular[start:text_vokabular.index("\n\n", start)])

gnd:967279453 a skos:Concept;
  skos:altLabel "Learning on demand"@de,
    "Teleteaching"@de,
    "Virtuelles Lernen"@de,
    "Telelearning"@de,
    "Electronic Learning"@de,
    "Digitales Lernen"@de;
  skos:related gnd:1044874414,
    gnd:041264649,
    gnd:041808576,
    gnd:041808525;
  skos:prefLabel "E-Learning"@de;
  skos:broader gnd:04225938X.


### 1.2 Was steht insgesamt drin?

Wir zählen jede Eigenschaft über die ganze Datei und markieren, was `build.py` daraus übernimmt.

In [3]:
eigenschaften = Counter(re.findall(r"^\s*(skos:\w+)", text_vokabular, re.M))

VERWENDUNG_1 = {
    "skos:prefLabel":   "✅ Name (begriff)",
    "skos:broader":     "✅ Oberbegriffe, umgedreht auch Unterbegriffe",
    "skos:related":     "✅ verwandte Begriffe",
    "skos:altLabel":    "nur gezählt (g['alt']), nicht angezeigt. Im Mapping von KI-FS Prompt genutzt",
    "skos:hiddenLabel": "nur gezählt, nicht angezeigt",
}
pd.DataFrame(
    [(e, f"{n:,}".replace(",", "."), VERWENDUNG_1.get(e, "❌ nicht verwendet")) for e, n in eigenschaften.most_common()],
    columns=["Eigenschaft", "Begriffe damit", "in gnd_context"],
)

,Eigenschaft,Begriffe damit,in gnd_context
0,skos:prefLabel,309.417,✅ Name (begriff)
1,skos:broader,193.860,"✅ Oberbegriffe, umgedreht auch Unterbegriffe"
2,skos:altLabel,178.816,"nur gezählt (g['alt']), nicht angezeigt. Im Ma..."
3,skos:related,140.294,✅ verwandte Begriffe
4,skos:hiddenLabel,43.155,"nur gezählt, nicht angezeigt"
5,skos:member,299,❌ nicht verwendet


Wo jede Eigenschaft im Code landet:

| Eigenschaft | Regex | wird zu | Code |
|---|---|---|---|
| Blockkopf `gnd:…` | `_ID` ([build.py:21](src/ie_agent/gnd/build.py#L21)) | eigene ID | [build.py:59](src/ie_agent/gnd/build.py#L59) |
| `skos:prefLabel` | `_LABEL` ([build.py:22](src/ie_agent/gnd/build.py#L22)) | `g["pref"]` | [build.py:64-67](src/ie_agent/gnd/build.py#L64-L67) |
| `skos:altLabel`, `skos:hiddenLabel` | `_LABEL` | `g["alt"]`, **nur die Anzahl** | [build.py:68-69](src/ie_agent/gnd/build.py#L68-L69) |
| `skos:broader` | `_ID` | `g["broader"]` | [build.py:70-71](src/ie_agent/gnd/build.py#L70-L71) |
| `skos:related` | `_ID` | `g["related"]`, `g["related_out"]` | [build.py:72-73](src/ie_agent/gnd/build.py#L72-L73), [build.py:81-86](src/ie_agent/gnd/build.py#L81-L86) |
| *(nicht in der Datei)* | – | `g["narrower"]`: `broader` umgedreht | [build.py:75-79](src/ie_agent/gnd/build.py#L75-L79) |

Das sind **alle** Eigenschaften der Datei. Es gibt **keine** Definitionen, Sachgruppen oder Hinweise und auch kein `skos:narrower`. (`skos:member` steht nur bei Sammlungen, und die sind keine Begriffe.)

### 1.3 Wie vollständig ist der Graph?

In [4]:
ohne_kante = sum(1 for i in ALLE if not (g["broader"].get(i) or g["narrower"].get(i) or g["related"].get(i)))
kanten_oben = sum(len(v) for v in g["broader"].values())

pd.DataFrame([
    ("Name (prefLabel)",                   anteil(len(ALLE))),
    ("mind. 1 Oberbegriff",                anteil(sum(1 for i in ALLE if g["broader"].get(i)))),
    ("mind. 1 Unterbegriff",               anteil(sum(1 for i in ALLE if g["narrower"].get(i)))),
    ("mind. 1 verwandter Begriff (eigene Richtung)", anteil(sum(1 for i in ALLE if g["related_out"].get(i)))),
    ("mind. 1 Synonym",                    anteil(sum(1 for i in ALLE if g["alt"].get(i)))),
    ("ganz ohne Kante (isoliert)",         anteil(ohne_kante)),
], columns=["", "Begriffe"]).style.hide(axis="index")

,Begriffe
Name (prefLabel),309.417 (100.0%)
mind. 1 Oberbegriff,193.860 (62.7%)
mind. 1 Unterbegriff,39.216 (12.7%)
mind. 1 verwandter Begriff (eigene Richtung),140.294 (45.3%)
mind. 1 Synonym,193.864 (62.7%)
ganz ohne Kante (isoliert),36.013 (11.6%)


In [5]:
print(f"Oberbegriff-Kanten insgesamt: {kanten_oben:,}".replace(",", "."))

Oberbegriff-Kanten insgesamt: 224.324


- **Vollständig:** Jeder Begriff hat eine ID und einen Namen.
- **Lückenhaft:** Etwa jeder dritte Begriff hat keinen Oberbegriff, etwa jeder neunte gar keine Kante. Bei diesen Begriffen kann `gnd_context` keine Hierarchie zeigen.
- Nur 13 % haben Unterbegriffe. Die meisten Begriffe sind also „Blätter“ ganz unten im Baum.

## Teil 2 · Quelle 2: die DNB-Dumps

Zwei Dateien aus DNB Open Data: **Sachbegriffe** und **Geografika** (Orte). Sie sind viel reicher als das Vokabular, kennen aber andere IDs (GND-Nummern wie `4727098-6` statt IDNs wie `967279453`). Gelesen werden sie **zeilenweise** von [_read_dumps, build.py:115-153](src/ie_agent/gnd/build.py#L115-L153), weil ein Begriff über mehrere Blöcke verteilt ist.

### 2.1 Roh: der Eintrag von E-Learning

Ein Begriff ist im Dump auf mehrere Blöcke verteilt. Wir sammeln alle Blöcke, die mit seiner GND-Nummer beginnen. Die `/about`-Verwaltungsblöcke gehören nicht dazu. In `build.py` sorgt dafür das `>` am Ende von `_SUBJECT` ([build.py:25](src/ie_agent/gnd/build.py#L25)): Bei `…/4727098-6/about>` folgt auf die Nummer ein `/`, also kein Treffer, und die Zeile wechselt nicht den aktuellen Begriff ([build.py:126-128](src/ie_agent/gnd/build.py#L126-L128)).

In [6]:
def eintrag_im_dump(gnd_nr):
    # Alle Blöcke, die mit <https://d-nb.info/gnd/NR> beginnen, aus beiden Dumps.
    kopf = f"<https://d-nb.info/gnd/{gnd_nr}> "
    blöcke, aktuell, gefunden = [], None, False
    for datei in DUMPS.values():
        for zeile in gzip.open(datei, "rt", encoding="utf-8"):
            if zeile.startswith(kopf):
                aktuell, gefunden = [zeile.rstrip()], True
            elif aktuell is not None:
                if zeile.strip():
                    aktuell.append(zeile.rstrip())
                else:
                    blöcke.append("\n".join(aktuell)); aktuell = None
            elif gefunden and zeile.startswith("<https://d-nb.info/gnd/") and gnd_nr not in zeile:
                return blöcke          # der nächste Begriff beginnt: fertig
    return blöcke

gnd_nr = details[BEISPIEL]["gnd_nr"]
print(f"E-Learning: IDN {BEISPIEL} = GND-Nummer {gnd_nr}\n")
print("\n\n".join(eintrag_im_dump(gnd_nr)))

E-Learning: IDN 967279453 = GND-Nummer 4727098-6

<https://d-nb.info/gnd/4727098-6> a gndo:SubjectHeadingSensoStricto;
  wdrs:describedby <https://d-nb.info/gnd/4727098-6/about> .

<https://d-nb.info/gnd/4727098-6> skos:exactMatch <http://zbw.eu/stw/descriptor/18074-2>;
  gndo:gndIdentifier "4727098-6";
  owl:sameAs <http://www.wikidata.org/entity/Q182250>;
  gndo:oldAuthorityNumber "(DE-588c)4727098-6";
  gndo:relatedDdcWithDegreeOfDeterminacy3 <http://dewey.info/class/371.358/>, <http://dewey.info/class/371.3344678/>,
    <http://dewey.info/class/374.26/>, <http://dewey.info/class/378.17344678/>;
  gndo:variantNameForTheSubjectHeading "Learning on demand", "Telelearning", "Teleteaching",
    "Electronic Learning", "Virtuelles Lernen", "Digitales Lernen";
  gndo:preferredNameForTheSubjectHeading "E-Learning";
  skos:closeMatch <http://id.loc.gov/authorities/subjects/sh85029483>, <https://ark.bnf.fr/ark:/12148/cb11931250t>;
  skos:exactMatch <http://zbw.eu/stw/descriptor/18074-2>, thes

### 2.2 Was steht insgesamt drin?

Gezählt wird, **wie viele Begriffe** eine Eigenschaft haben, getrennt nach Dump. Dazu kommt, was `build.py` davon übernimmt. Diese Zelle liest beide Dumps komplett.

In [7]:
SUBJEKT = re.compile(r"<https://d-nb.info/gnd/([0-9X-]+)> ")
EIGENSCHAFT = re.compile(r"^(?:<[^>]+>\s+|\s+)(\w+:\w+)")
NAME = re.compile(r'gndo:preferredNameFor\w+ "((?:[^"\\]|\\.)*)"')

pro_dump = {}
dump_namen = {}                      # Name -> Liste der Dumps, in denen er vorkommt
for art, datei in DUMPS.items():
    paare = set()
    aktuell = None
    for zeile in gzip.open(datei, "rt", encoding="utf-8"):
        m = SUBJEKT.match(zeile)
        if m:
            aktuell = m.group(1)
        e = EIGENSCHAFT.match(zeile)
        if e and aktuell:
            paare.add((aktuell, e.group(1)))
        n = NAME.search(zeile)
        if n:
            dump_namen.setdefault(nfc(n.group(1)), []).append(art)
    pro_dump[art] = Counter(e for _, e in paare)

VERWENDUNG_2 = {
    "gndo:preferredNameForTheSubjectHeading":         "✅ intern: Zuordnung zur IDN über den Namen",
    "gndo:preferredNameForThePlaceOrGeographicName":  "✅ intern: Zuordnung zur IDN über den Namen",
    "gndo:definition":                                "✅ definition",
    "gndo:usingInstructions":                         "✅ benutzungshinweis",
    "gndo:biographicalOrHistoricalInformation":       "✅ beschreibung",
    "gndo:gndSubjectCategory":                        "✅ sachgruppen (Code, Name aus Quelle 3)",
    "gndo:broaderTermGeneric":                        "✅ beziehung: Art von",
    "gndo:broaderTermPartitive":                      "✅ beziehung: Teil von",
    "gndo:broaderTermInstantial":                     "✅ beziehung: Instanz von",
    "gndo:broaderTermGeneral":                        "– Art unbekannt, kein Etikett",
}
alle_eigenschaften = sorted(set(pro_dump["Sachbegriff"]) | set(pro_dump["Geografikum"]),
                            key=lambda e: -(pro_dump["Sachbegriff"][e] + pro_dump["Geografikum"][e]))
tabelle_2 = pd.DataFrame(
    [(e, pro_dump["Sachbegriff"][e], pro_dump["Geografikum"][e], VERWENDUNG_2.get(e, "❌")) for e in alle_eigenschaften],
    columns=["Eigenschaft", "Sachbegriffe", "Geografika", "in gnd_context"],
)
print(f"Begriffe in den Dumps: {sum(len(v) for v in dump_namen.values()):,}".replace(",", "."))
tabelle_2.head(35)

Begriffe in den Dumps: 542.201


,Eigenschaft,Sachbegriffe,Geografika,in gnd_context
0,gndo:gndIdentifier,207505,334696,❌
1,dcterms:license,207505,334696,❌
2,dcterms:modified,207505,334696,❌
3,wdrs:describedby,207505,334696,❌
4,gndo:descriptionLevel,207505,334696,❌
5,dcterms:creator,207502,334680,❌
6,dcatde:maintainer,207502,334680,❌
7,gndo:oldAuthorityNumber,184800,247259,❌
8,gndo:geographicAreaCode,28592,334353,❌
9,owl:sameAs,39302,300959,❌


**Was man sieht:**

- Die Dumps sind **sehr viel reicher** als das Vokabular: Synonyme, DDC-Klassen, Verknüpfungen zu Wikidata und LoC, Geokoordinaten, Gründungsdaten, Vorgänger und Nachfolger von Orten …
- `gnd_context` nimmt davon bewusst nur das, was beim **Prüfen** eines Kandidaten hilft. Wo das im Code passiert:

| Eigenschaft | Regex / Code | wird zu |
|---|---|---|
| `gndo:preferredNameFor…` | `_NAME` [build.py:26](src/ie_agent/gnd/build.py#L26), gelesen [build.py:141-143](src/ie_agent/gnd/build.py#L141-L143) | Name, nur für die Zuordnung zur IDN |
| `gndo:definition`, `usingInstructions`, `biographicalOrHistoricalInformation` | `TEXT_FIELDS` [build.py:31-35](src/ie_agent/gnd/build.py#L31-L35), `_LITERAL` [build.py:28](src/ie_agent/gnd/build.py#L28), gelesen [build.py:145-148](src/ie_agent/gnd/build.py#L145-L148) | `definition`, `using_instructions`, `description` |
| `gndo:gndSubjectCategory` | [build.py:150-151](src/ie_agent/gnd/build.py#L150-L151) | Sachgruppen-Codes |
| `gndo:broaderTermGeneric/Partitive/Instantial` | `_BROADER_KIND` [build.py:27](src/ie_agent/gnd/build.py#L27), gelesen [build.py:131-139](src/ie_agent/gnd/build.py#L131-L139) | Art pro Oberbegriff |
- Mit `gndo:relatedDdcWithDegreeOfDeterminacy…` (DDC-Klasse) oder `gndo:geographicAreaCode` stünden noch weitere Prüf-Signale bereit. Die bleiben bisher ungenutzt.

### 2.3 Wie viel kommt beim Vokabular an? Die Zuordnung über den Namen

Ein Dump-Eintrag nützt nur, wenn er einer IDN im Vokabular zugeordnet werden kann. Eine ID-Tabelle gibt es nicht, deshalb ordnet [_parse_details, build.py:162-177](src/ie_agent/gnd/build.py#L162-L177) **über den Vorzugsnamen** zu, und nur wenn der Name im Vokabular **und** in den Dumps genau einmal vorkommt ([build.py:171](src/ie_agent/gnd/build.py#L171)). Vorher werden alle Namen mit `nfc` vereinheitlicht ([build.py:38-41](src/ie_agent/gnd/build.py#L38-L41)), weil die Dumps viele Umlaute zerlegt speichern.

In [8]:
namen_im_vokabular = Counter(nfc(n) for n in g["pref"].values())
grund, dump_art = Counter(), {}
for i in ALLE:
    n = nfc(g["pref"][i])
    if i in details:
        grund["✅ zugeordnet"] += 1
        dump_art[i] = dump_namen[n][0]
    elif n not in dump_namen:
        grund["❌ Name in keinem der beiden Dumps"] += 1
    elif namen_im_vokabular[n] > 1:
        grund["❌ Name im Vokabular mehrfach"] += 1
    else:
        grund["❌ Name in den Dumps mehrfach"] += 1

display(pd.DataFrame([(k, anteil(v)) for k, v in grund.most_common()], columns=["Vokabular-Begriffe", ""]))
print("zugeordnet nach Dump:", {k: f"{v:,}".replace(",", ".") for k, v in Counter(dump_art.values()).items()})

,Vokabular-Begriffe,
0,✅ zugeordnet,197.252 (63.7%)
1,❌ Name in keinem der beiden Dumps,111.719 (36.1%)
2,❌ Name in den Dumps mehrfach,386 (0.1%)
3,❌ Name im Vokabular mehrfach,60 (0.0%)


zugeordnet nach Dump: {'Sachbegriff': '173.385', 'Geografikum': '23.867'}


In [9]:
import random
random.seed(1)
nicht_im_dump = [g["pref"][i] for i in ALLE if i not in details and nfc(g["pref"][i]) not in dump_namen]
random.sample(nicht_im_dump, 15)

['Godefridus (de Franconia) : Pelzbuch',
 'Komplement (Immunologie)',
 'River Clyde (Schiff)',
 'Würde (Astrologie)',
 'Kepler, Katharina (1547-1622)',
 'Johannes-Apokalypse / Kommentar / Oxford / Bodleian Library / Ms. Bodley 352',
 'Erwachsener (30 Jahre)',
 'Bogen (Streichinstrument)',
 'Arbeitsmotivation (Motiv)',
 'Munch, Jacob Edvard (1776-1839)',
 'Bruni, Bruno (1935)',
 'Der Himmel über Berlin',
 'Biden, Joseph R. (1942)',
 'Mann, Thomas : Mario und der Zauberer',
 'Jordaens, Wilhelm : Conflictus virtutum et viciorum']

**Die größte Lücke:** Gut ein Drittel des Vokabulars steht in keinem der beiden Dumps. Die Stichprobe zeigt, warum: Das sind **Personen** („Buck, Pearl S. (1892-1973)“), **Körperschaften** (Firmen, Institute), **Werke** („Seiler, Lutz : Kruso“) und Familien. Für die gibt es eigene DNB-Dumps (Personen, Körperschaften, Werke), die `build.py` nicht lädt. Diese Begriffe bekommen in `gnd_context` deshalb **nie** Definition, Sachgruppe oder Beziehungsart.

Die Fälle mit mehrfachem Namen sind dagegen selten (unter 0,2 %). Die strenge Eindeutigkeitsregel kostet also fast nichts.

### 2.4 Welche Felder kommen an?

In [10]:
kanten = [(kind, eltern) for kind, ps in g["broader"].items() for eltern in ps]
arten = Counter(details.get(k, {}).get("broader_types", {}).get(e, "keine Art (General oder nicht zugeordnet)")
                for k, e in kanten)

display(pd.DataFrame([
    ("definition",        anteil(sum(1 for v in details.values() if v["definition"]))),
    ("benutzungshinweis", anteil(sum(1 for v in details.values() if v["using_instructions"]))),
    ("beschreibung",      anteil(sum(1 for v in details.values() if v["description"]))),
    ("sachgruppe (mit Namen)", anteil(sum(1 for i in details if gnd.categories(i)))),
], columns=["Feld", "von allen Vokabular-Begriffen"]))

display(pd.DataFrame([(a, anteil(n, len(kanten))) for a, n in arten.most_common()],
                     columns=["Art der Oberbegriff-Kante", "von allen Kanten"]))

,Feld,von allen Vokabular-Begriffen
0,definition,47.127 (15.2%)
1,benutzungshinweis,10.951 (3.5%)
2,beschreibung,14.449 (4.7%)
3,sachgruppe (mit Namen),181.750 (58.7%)


,Art der Oberbegriff-Kante,von allen Kanten
0,keine Art (General oder nicht zugeordnet),154.322 (68.8%)
1,Generic,32.860 (14.6%)
2,Instantial,27.386 (12.2%)
3,Partitive,9.756 (4.3%)


- **Sachgruppe** ist das einzige Dump-Feld, das die Mehrheit der Begriffe hat.
- **Definition** und **Benutzungshinweis** sind selten.
- Bei **Beschreibung** fällt etwas auf: Im Geografikum-Dump haben über die Hälfte der Orte eine (Tabelle 2.2), im Vokabular kommen aber nur 5 % an. Die meisten Orte der GND sind gar nicht im Vokabular.
- Nur knapp ein Drittel der Oberbegriff-Kanten bekommt ein **Etikett** (Art, Teil, Instanz). Der Rest steht ohne `beziehung` da. Die Art wird nur übernommen, wenn **beide** Seiten der Kante zugeordnet wurden ([build.py:180-186](src/ie_agent/gnd/build.py#L180-L186)). In `gnd_context` hängt `relation()` sie an, wenn sie vorhanden ist ([context.py:97-100](src/ie_agent/gnd/context.py#L97-L100)), und übersetzt sie in einen Satz ([context.py:5-14](src/ie_agent/gnd/context.py#L5-L14)).

## Teil 3 · Quelle 3: die Sachgruppen `gnd-sc.ttl`

Eine kleine Liste: **Code → Name** in Deutsch und Englisch. Sie kennt keinen einzigen Begriff wie E-Learning, sie übersetzt nur die Codes aus Quelle 2.

- Gelesen von [_category_names, build.py:102-112](src/ie_agent/gnd/build.py#L102-L112). Die Labels stehen in beliebiger Reihenfolge (`@en`, `@de`), deshalb wird gezielt das deutsche gesucht ([build.py:109](src/ie_agent/gnd/build.py#L109)).
- Übersetzt wird beim Speichern der Details ([build.py:176](src/ie_agent/gnd/build.py#L176)): `category_names.get(code, code)`, also Name, sonst der Code.
- Codes ohne Namen filtert `Gnd.categories` heraus ([graph.py:80-83](src/ie_agent/gnd/graph.py#L80-L83), Muster `CATEGORY_CODE` [graph.py:15](src/ie_agent/gnd/graph.py#L15)).

In [11]:
codes_e_learning = re.findall(r"gnd-sc#([^>]+)>", "\n".join(eintrag_im_dump(gnd_nr)))
text_sc = SC.read_text(encoding="utf-8")
for c in dict.fromkeys(codes_e_learning):
    start = text_sc.index(f"<https://d-nb.info/standards/vocab/gnd/gnd-sc#{c}>\n")
    print(text_sc[start:text_sc.index("\n\n", start)], "\n")

<https://d-nb.info/standards/vocab/gnd/gnd-sc#6.2a>
    a skos:Concept ;
    skos:inScheme <https://d-nb.info/standards/vocab/gnd/gnd-sc#> ;
    skos:notation "6.2a" ;
    skos:prefLabel "Bildungswesen (Allgemeines)"@de, "Education (misc.)"@en . 

<https://d-nb.info/standards/vocab/gnd/gnd-sc#30>
    a skos:Concept ;
    skos:inScheme <https://d-nb.info/standards/vocab/gnd/gnd-sc#> ;
    skos:notation "30" ;
    skos:prefLabel "Computer science, data processing"@en, "Informatik, Datenverarbeitung"@de . 



In [12]:
sachgruppen = _category_names(SC)
benutzt = Counter(c for v in details.values() for c in v["classification"])
ohne_namen = {c: n for c, n in benutzt.items() if re.match(r"^[0-9.]+[a-z]*$", c)}

print(f"Sachgruppen in gnd-sc.ttl         : {len(sachgruppen)}")
print(f"davon bei unseren Begriffen benutzt: {len([c for c in benutzt if c not in ohne_namen])}")
print(f"Codes ohne Namen (werden ausgefiltert): {ohne_namen}")

Sachgruppen in gnd-sc.ttl         : 483
davon bei unseren Begriffen benutzt: 341
Codes ohne Namen (werden ausgefiltert): {'10.2dc': 117}


- **Vollständig:** Bis auf einen einzigen Code (`10.2dc`) hat jede benutzte Sachgruppe einen Namen. Quelle 3 ist also praktisch lückenlos.
- Ihr Beitrag ist klein, aber wichtig: Ohne sie sähe der Agent `31.9b` statt „Elektronik, Kommunikationstechnik“.

## Teil 4 · Kein GND-Feld: die Häufigkeit im Training

`in_trainingsdokumenten_vergeben` stammt aus **keiner** GND-Datei. Es wird aus dem Gold der Trainingsdokumente gezählt (`training_frequency`, [graph.py:30](src/ie_agent/gnd/graph.py#L30)). Das Validierungs-Gold wird dafür nie verwendet, sonst wäre es Leakage.

In [13]:
f = gnd.frequency
print(f"Begriffe, die im Training mind. 1-mal vergeben wurden: {anteil(sum(1 for i in ALLE if f.get(i)))}")
print("am häufigsten:", [(g['pref'].get(i, '?'), n) for i, n in sorted(f.items(), key=lambda x: -x[1])[:8]])

Begriffe, die im Training mind. 1-mal vergeben wurden: 17.302 (5.6%)
am häufigsten: [('Deutschland', 549), ('Tiermodell', 289), ('Finite-Elemente-Methode', 152), ('Prüfstand', 133), ('Hund', 123), ('Produktentwicklung', 107), ('Simulation', 105), ('Herstellung', 103)]


## Teil 5 · Die fertige Antwort, jede Zeile mit ihrer Quelle

Szenario (ausgedacht): Der Agent hat den Titel *„Webinare in der Lehrerfortbildung: Erfahrungen mit E-Learning an Schulen“* gelesen, **E-Learning** und **Computerunterstütztes Lernen** abgebildet und fragt `gnd_context` nach E-Learning.

Die Antwort baut [gnd_context, context.py:73-133](src/ie_agent/gnd/context.py#L73-L133). Der Agent bekommt sie als JSON-Text über [tools.py:343-350](src/ie_agent/tools.py#L343-L350).

Markierung: **[1]** Vokabular · **[2]** DNB-Dump · **[3]** gnd-sc.ttl · **[G]** Trainings-Gold · **[L]** erst zur Laufzeit berechnet

In [14]:
state = {
    "text": ["Webinare in der Lehrerfortbildung: Erfahrungen mit E-Learning an Schulen"],
    "mapped": {name_zu_id["E-Learning"]: {}, name_zu_id["Computerunterstütztes Lernen"]: {}},
    "graph": {},
}
antwort = gnd_context(gnd, BEISPIEL, state, mapper=None, max_narrower=8)   # mapper nur bei > 8 Unterbegriffen nötig

HERKUNFT = {
    ("gnd_id", "gnd_id"):                 "[1] Blockkopf gnd:…",
    ("begriff", "begriff"):               "[1] skos:prefLabel",
    ("definition", "definition"):         "[2] gndo:definition (gekürzt auf 350 Zeichen)",
    ("benutzungshinweis", "benutzungshinweis"): "[2] gndo:usingInstructions",
    ("beschreibung", "beschreibung"):     "[2] gndo:biographicalOrHistoricalInformation",
    ("sachgruppen", "sachgruppen"):       "[2] gndo:gndSubjectCategory (Code) + [3] Name",
    ("sachgruppen", None):                "[3] Name des Codes",
    ("in_trainingsdokumenten_vergeben", "in_trainingsdokumenten_vergeben"): "[G] gezählt im Trainings-Gold",
    ("oberbegriffe", "oberbegriffe"):     "[1] skos:broader (max. 5)",
    ("oberbegriffe", "gnd_id"):           "[1] skos:broader",
    ("oberbegriffe", "begriff"):          "[1] skos:prefLabel des Oberbegriffs",
    ("oberbegriffe", "beziehung"):        "[2] gndo:broaderTermGeneric/Partitive/Instantial",
    ("unterbegriffe", "unterbegriffe"):   "[1] skos:broader umgedreht",
    ("unterbegriffe", "anzahl"):          "[1] alle Unterbegriffe",
    ("unterbegriffe", "auswahl"):         "[L] max. 8: im Text → ähnlich → häufig",
    ("unterbegriffe", "gnd_id"):          "[1] hat den Kandidaten als skos:broader",
    ("unterbegriffe", "begriff"):         "[1] skos:prefLabel des Unterbegriffs",
    ("unterbegriffe", "beziehung"):       "[2] broaderTerm… im Eintrag des Unterbegriffs",
    ("unterbegriffe", "im_text"):         "[L] Name steht im gelesenen Text",
    ("verwandte_begriffe", "verwandte_begriffe"): "[1] skos:related, eigene Richtung (max. 5)",
    ("verwandte_begriffe", "gnd_id"):     "[1] skos:related",
    ("verwandte_begriffe", "begriff"):    "[1] skos:prefLabel des verwandten Begriffs",
    ("beziehungen_zu_deinen_kandidaten", "beziehungen_zu_deinen_kandidaten"): "[L] [1]-Graph × state['mapped']",
    ("beziehungen_zu_deinen_kandidaten", None): "[L] berechnet, steht in keiner Datei",
}

abschnitt = None
for zeile in json.dumps(antwort, ensure_ascii=False, indent=2).splitlines():
    m = re.match(r'\s*"(\w+)":', zeile)
    feld = m.group(1) if m else None
    if feld and len(zeile) - len(zeile.lstrip()) == 2:
        abschnitt = feld
    ist_wert = zeile.strip().startswith('"') and not feld
    quelle = HERKUNFT.get((abschnitt, feld)) if feld else (HERKUNFT.get((abschnitt, None)) if ist_wert else None)
    text = zeile if len(zeile) <= 60 else zeile[:57] + "…"
    print(f"{text:60s}" + (f"  ← {quelle}" if quelle else ""))

{                                                           
  "gnd_id": "967279453",                                      ← [1] Blockkopf gnd:…
  "begriff": "E-Learning",                                    ← [1] skos:prefLabel
  "definition": "Lernen unter Nutzung der im Netz verfügb…    ← [2] gndo:definition (gekürzt auf 350 Zeichen)
  "benutzungshinweis": "Verknüpfe mit den jeweiligen SWW …    ← [2] gndo:usingInstructions
  "sachgruppen": [                                            ← [2] gndo:gndSubjectCategory (Code) + [3] Name
    "Bildungswesen (Allgemeines)",                            ← [3] Name des Codes
    "Informatik, Datenverarbeitung"                           ← [3] Name des Codes
  ],                                                        
  "in_trainingsdokumenten_vergeben": 31,                      ← [G] gezählt im Trainings-Gold
  "oberbegriffe": [                                           ← [1] skos:broader (max. 5)
    {                                             

**Jedes Feld der Antwort im Code:**

| Feld | Quelle | gebaut in `build.py` | ausgegeben in `context.py` |
|---|---|---|---|
| `gnd_id`, `begriff` | [1] | [build.py:59](src/ie_agent/gnd/build.py#L59), [build.py:64-67](src/ie_agent/gnd/build.py#L64-L67) | [context.py:109-110](src/ie_agent/gnd/context.py#L109-L110), Name über `Gnd.label` [graph.py:77-78](src/ie_agent/gnd/graph.py#L77-L78) |
| `definition` | [2] | [build.py:145-148](src/ie_agent/gnd/build.py#L145-L148) | [context.py:111](src/ie_agent/gnd/context.py#L111), gekürzt mit `_short` [context.py:26-27](src/ie_agent/gnd/context.py#L26-L27) auf `MAX_TEXT` [context.py:17](src/ie_agent/gnd/context.py#L17) |
| `benutzungshinweis`, `beschreibung` | [2] | [build.py:145-148](src/ie_agent/gnd/build.py#L145-L148) | nur wenn vorhanden: [context.py:103-106](src/ie_agent/gnd/context.py#L103-L106), [context.py:112](src/ie_agent/gnd/context.py#L112) |
| `sachgruppen` | [2] + [3] | [build.py:150-151](src/ie_agent/gnd/build.py#L150-L151), [build.py:176](src/ie_agent/gnd/build.py#L176) | [context.py:113](src/ie_agent/gnd/context.py#L113) über `Gnd.categories` [graph.py:80-83](src/ie_agent/gnd/graph.py#L80-L83) |
| `in_trainingsdokumenten_vergeben` | [G] | `training_frequency` [graph.py:30-34](src/ie_agent/gnd/graph.py#L30-L34) | [context.py:114](src/ie_agent/gnd/context.py#L114) |
| `oberbegriffe` | [1] | [build.py:70-71](src/ie_agent/gnd/build.py#L70-L71) | max. 5, häufigste zuerst: [context.py:84-86](src/ie_agent/gnd/context.py#L84-L86), [context.py:115-118](src/ie_agent/gnd/context.py#L115-L118) |
| `oberbegriffe[].beziehung` | [2] | [build.py:131-139](src/ie_agent/gnd/build.py#L131-L139), [build.py:180-186](src/ie_agent/gnd/build.py#L180-L186) | [context.py:97-100](src/ie_agent/gnd/context.py#L97-L100), Text [context.py:5-9](src/ie_agent/gnd/context.py#L5-L9) |
| `unterbegriffe` | [1] | umgedreht [build.py:75-79](src/ie_agent/gnd/build.py#L75-L79) | Auswahl [context.py:30-52](src/ie_agent/gnd/context.py#L30-L52), Ausgabe [context.py:119-130](src/ie_agent/gnd/context.py#L119-L130) |
| `unterbegriffe[].im_text` | [L] | – | [context.py:39-42](src/ie_agent/gnd/context.py#L39-L42) |
| `unterbegriffe[].beziehung` | [2] | [build.py:180-186](src/ie_agent/gnd/build.py#L180-L186) | [context.py:97-100](src/ie_agent/gnd/context.py#L97-L100), Text [context.py:10-14](src/ie_agent/gnd/context.py#L10-L14) |
| `verwandte_begriffe` | [1] | [build.py:72-73](src/ie_agent/gnd/build.py#L72-L73), `related_out` [build.py:81-83](src/ie_agent/gnd/build.py#L81-L83) | max. 5: [context.py:88-90](src/ie_agent/gnd/context.py#L88-L90), [context.py:131](src/ie_agent/gnd/context.py#L131) |
| `beziehungen_zu_deinen_kandidaten` | [L] | – | `_relations_to_candidates` [context.py:55-70](src/ie_agent/gnd/context.py#L55-L70), [context.py:132](src/ie_agent/gnd/context.py#L132) |
| *(Seiteneffekt)* gezeigte Begriffe dürfen in `finish` vergeben werden | [L] | – | [context.py:93-94](src/ie_agent/gnd/context.py#L93-L94) |

## Teil 6 · Statistik: was jede Quelle beiträgt, was vollständig ist, was fehlt

Bisher haben wir über **alle** 309.417 Begriffe gezählt. Für den Agenten zählt aber, wie gut die Information bei den Begriffen ist, die **richtig** wären. Deshalb zählen wir zusätzlich über die **Gold-Vergaben** der Validierungsdokumente, die auch die Agent-Runs benutzen.

Das ist eine reine Analyse nach dem Run. Zur Laufzeit sieht der Agent das Validierungs-Gold nie.

In [15]:
log = ROOT / "ie_agnet/results/Qwen3.8-27B-FP8/reasoning-off_gnd_context-on_toc-off_examples-optional/log.jsonl"
dokumente = {json.loads(z)["doc_id"] for z in open(log, encoding="utf-8")}

gt = pyarrow.ipc.open_file(ROOT / "corpora/ground-truth.arrow").read_all().to_pandas().astype(str)
gt = gt[(gt["kind"] == "title") & gt["idn"].isin(dokumente)]
gold = [u.split("/")[-1] for u in gt["uri"]]
print(f"{len(dokumente)} Dokumente · {len(gold):,} Gold-Vergaben · {len(set(gold)):,} verschiedene Schlagwörter".replace(",", "."))

827 Dokumente · 4.018 Gold-Vergaben · 3.134 verschiedene Schlagwörter


/tmp/ipykernel_4060145/289577216.py:2: ResourceWarning: unclosed file <_io.TextIOWrapper name='/home/harbc/ki-fsprompt/ie_agnet/results/Qwen3.8-27B-FP8/reasoning-off_gnd_context-on_toc-off_examples-optional/log.jsonl' mode='r' encoding='utf-8'>
  dokumente = {json.loads(z)["doc_id"] for z in open(log, encoding="utf-8")}


In [16]:
def hat_art(i):
    return any(details.get(i, {}).get("broader_types", {}).get(p) for p in g["broader"].get(i, ()))

FELDER = [
    # (Feld in gnd_context, Quelle, Test pro Begriff)
    ("gnd_id, begriff",                   "[1] Vokabular",  lambda i: i in g["pref"]),
    ("oberbegriffe",                      "[1] Vokabular",  lambda i: bool(g["broader"].get(i))),
    ("unterbegriffe",                     "[1] Vokabular",  lambda i: bool(g["narrower"].get(i))),
    ("verwandte_begriffe",                "[1] Vokabular",  lambda i: bool(g["related_out"].get(i))),
    ("überhaupt ein Dump-Eintrag",        "[2] DNB-Dump",   lambda i: i in details),
    ("definition",                        "[2] DNB-Dump",   lambda i: bool(details.get(i, {}).get("definition"))),
    ("benutzungshinweis",                 "[2] DNB-Dump",   lambda i: bool(details.get(i, {}).get("using_instructions"))),
    ("beschreibung",                      "[2] DNB-Dump",   lambda i: bool(details.get(i, {}).get("description"))),
    ("beziehung bei mind. 1 Oberbegriff", "[2] DNB-Dump",   hat_art),
    ("sachgruppen",                       "[2] + [3]",      lambda i: bool(gnd.categories(i))),
    ("in_trainingsdokumenten_vergeben > 0", "[G] Training", lambda i: bool(gnd.frequency.get(i))),
]

statistik = pd.DataFrame(
    [(feld, quelle,
      sum(test(i) for i in ALLE) / N,
      sum(test(i) for i in gold) / len(gold))
     for feld, quelle, test in FELDER],
    columns=["Feld", "Quelle", "alle Vokabular-Begriffe", "Gold-Vergaben (Validierung)"],
)
statistik.style.format({"alle Vokabular-Begriffe": "{:.1%}", "Gold-Vergaben (Validierung)": "{:.1%}"}) \
    .bar(subset=["alle Vokabular-Begriffe", "Gold-Vergaben (Validierung)"], vmin=0, vmax=1, color="#9ecae1") \
    .hide(axis="index")

Feld,Quelle,alle Vokabular-Begriffe,Gold-Vergaben (Validierung)
"gnd_id, begriff",[1] Vokabular,100.0%,98.9%
oberbegriffe,[1] Vokabular,62.7%,65.2%
unterbegriffe,[1] Vokabular,12.7%,63.4%
verwandte_begriffe,[1] Vokabular,45.3%,47.8%
überhaupt ein Dump-Eintrag,[2] DNB-Dump,63.7%,90.9%
definition,[2] DNB-Dump,15.2%,17.3%
benutzungshinweis,[2] DNB-Dump,3.5%,18.9%
beschreibung,[2] DNB-Dump,4.7%,1.7%
beziehung bei mind. 1 Oberbegriff,[2] DNB-Dump,19.8%,23.7%
sachgruppen,[2] + [3],58.7%,85.2%


### Gold-Schlagwörter ohne Dump-Eintrag: wer ist das?

In [17]:
ohne = [g["pref"].get(i, "?") for i in set(gold) if i in ALLE and i not in details]
print(f"{len(ohne)} verschiedene Gold-Schlagwörter ohne Dump-Eintrag, z. B.:")
random.seed(2)
random.sample(ohne, 15)

293 verschiedene Gold-Schlagwörter ohne Dump-Eintrag, z. B.:


['Stiftung Marktwirtschaft',
 'Wittgenstein, Ludwig : Tractatus logico-philosophicus',
 'Katholische Kirche / Curia Romana',
 'Erziehung (Motiv)',
 'Lanzelot (Mittelhochdeutsch, Prosa)',
 'Thomas (von Aquin, Heiliger) (1225-1274)',
 'Frankfurter Allgemeine',
 'Albert-Ludwigs-Universität Freiburg / Theologische Fakultät',
 'Korpus (Linguistik)',
 'Isolierung (Chemie)',
 'Ontologie (Wissensverarbeitung)',
 'Krise (Motiv)',
 'Globalisierung (Motiv)',
 'Komposition (Musik)',
 'Schelling, Friedrich Wilhelm Joseph von (1775-1854)']

### Ein Befund: runde gegen spitze Klammern

In der Stichprobe oben stehen nicht nur Personen und Werke, sondern auch normale Sachbegriffe wie „Aufschluss (Chemie)“. Der Grund: Das Vokabular schreibt Zusätze in **runden** Klammern, die DNB-Dumps in **spitzen**:

| Vokabular | DNB-Dump |
|---|---|
| `Aufschluss (Chemie)` | `Aufschluss <Chemie>` |

Für den Namensabgleich in `_parse_details` ([build.py:162-171](src/ie_agent/gnd/build.py#L162-L171)) sind das zwei verschiedene Namen. Wie viele Begriffe betrifft das?

In [18]:
def spitz(name):
    # "Aufschluss (Chemie)" -> "Aufschluss <Chemie>" (nur der Zusatz am Ende)
    return re.sub(r" \(([^()]*)\)$", r" <\1>", name)

verloren = [i for i, n in g["pref"].items()
            if i not in details and nfc(n) not in dump_namen and spitz(nfc(n)) in dump_namen
            and len(dump_namen[spitz(nfc(n))]) == 1 and namen_im_vokabular[nfc(n)] == 1]
betroffen = set(verloren)

print(f"Vokabular-Begriffe, die mit <…> eindeutig zugeordnet würden: {anteil(len(verloren))}")
print(f"Gold-Vergaben davon betroffen: {anteil(sum(i in betroffen for i in gold), len(gold))}")
ohne_details = [i for i in gold if i in ALLE and i not in details]
print(f"Gold-Vergaben ohne Dump-Eintrag insgesamt: {len(ohne_details)}, davon nur wegen der Klammern: "
      f"{sum(i in betroffen for i in ohne_details)}")
print("Beispiele:", [g["pref"][i] for i in verloren[:8]])

Vokabular-Begriffe, die mit <…> eindeutig zugeordnet würden: 25.848 (8.4%)
Gold-Vergaben davon betroffen: 152 (3.8%)
Gold-Vergaben ohne Dump-Eintrag insgesamt: 323, davon nur wegen der Klammern: 152
Beispiele: ['Sicherheitspolitik (Motiv)', 'Injektor (Teilchenbeschleuniger)', 'Walda (Motiv)', 'Gourmand (Motiv)', 'Avenue (Programmiersprache)', "Ribera d'Ebre / Aufstand (1822)", 'Gimajt (Volk)', 'Bänderung (Geologie)']


Gut jeder zwölfte Vokabular-Begriff verliert seine Definition, Sachgruppe und Beziehungsart **nur wegen der Klammerschreibweise**. Bei den Gold-Vergaben ist das fast die Hälfte der Fälle ohne Dump-Eintrag. Der Rest sind Personen, Körperschaften und Werke.

Geändert ist im Code noch nichts. Eine Korrektur wäre klein (beim Abgleich `<…>` und `(…)` gleich behandeln), würde aber `build-gnd` und damit alle `gnd_context`-Ergebnisse verändern. Deshalb gehört sie als eigene, begründete Entscheidung ins Projekt, nicht still nebenbei.

## Fazit

**Vollständig (praktisch 100 %):**
- **ID und Name** aus Quelle 1, für jeden Begriff.
- **Sachgruppen-Namen** aus Quelle 3: Jeder benutzte Code bis auf einen ist übersetzt.

**Gut abgedeckt, vor allem bei den Gold-Schlagwörtern:**
- **Sachgruppe:** bei der großen Mehrheit der Gold-Vergaben. Das ist das zuverlässigste Prüf-Signal aus den Dumps.
- **Häufigkeit im Training:** Die meisten Gold-Schlagwörter wurden im Training schon einmal vergeben. Das ist ein starkes Signal, aber eins, das neue oder seltene Schlagwörter benachteiligt.
- **Ober- und Unterbegriffe:** bei rund zwei Dritteln der Gold-Vergaben. Gold-Schlagwörter sind deutlich besser vernetzt als der Durchschnitt des Vokabulars.

**Lückenhaft:**
- **Definition** und **Benutzungshinweis:** nur bei einem kleinen Teil. Benutzungshinweise sind bei Gold-Schlagwörtern aber viel häufiger als im Vokabular insgesamt. Gerade die oft vergebenen Begriffe haben also Regeln.
- **Beziehungsart:** nur bei einem Teil der Oberbegriff-Kanten. Der Rest ist `General` oder nicht zugeordnet.

**Fehlt, aber behebbar:**
- **Sachbegriffe mit Zusatz in Klammern** („Aufschluss (Chemie)“): Sie gehen beim Namensabgleich verloren, weil der Dump `<Chemie>` schreibt. Das betrifft etwa 8 % des Vokabulars und knapp 4 % der Gold-Vergaben.

**Fehlt ganz:**
- **Personen, Körperschaften, Werke:** Sie stehen im Vokabular, aber in keinem geladenen Dump. Ihre Dumps lädt `build.py` nicht, also bekommen sie nie Details. Zusammen mit den Klammer-Fällen ist das die Lücke „überhaupt ein Dump-Eintrag“ bei den Gold-Vergaben (rund 9 %).
- **Ungenutzt, obwohl vorhanden:** DDC-Klassen, Ländercodes und Synonyme aus den Dumps. Das sind mögliche weitere Prüf-Signale.

Für die Arbeit: Die Wirkung von `gnd_context` (+0.03 F1) entsteht, **obwohl** der Agent bei den meisten Kandidaten nur Name, Sachgruppe, Graph und Häufigkeit sieht, aber keine Definition. Mit der Klammer-Korrektur, den Personen- und Werk-Dumps oder weiteren Feldern ließe sich die Abdeckung erhöhen. Ob das hilft, wäre eine eigene Ablation.